# Setup

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
from vnstock3 import Vnstock

In [ ]:
symbol = "FPT"
stock = Vnstock().stock(symbol=symbol, source='TCBS')
df_quote_history= stock.quote.history(start='2018-01-01', end='2025-01-31')
df_quote_history.to_csv(f'new/{symbol}.csv', index=False)
print('done!')

In [ ]:
files = glob.glob("new/*.csv")

list_df = []
for file in files:
    df_temp = pd.read_csv(file)
    symbol = os.path.basename(file).split('.')[0]
    df_temp['Symbol'] = symbol
    list_df.append(df_temp)

df = pd.concat(list_df, ignore_index=True)

df['time'] = pd.to_datetime(df['time'])

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
len(df)

## Complete Data from 01/01/2018 to 31/01/2025

Fill data with linear interpolation

In [ ]:
full_range = pd.date_range(start='2018-01-01', end='2025-01-31', freq='D')
numeric_cols = ['open', 'high', 'low', 'close', 'volume']
filled_list = []

df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d')

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values('time').drop_duplicates(subset='time', keep='first')
    group = group.set_index('time')
    
    group = group.reindex(full_range)
    group.index.name = 'time'
    group['Symbol'] = symbol  
    
    group[numeric_cols] = group[numeric_cols].interpolate(method='linear').ffill().bfill()
    
    group = group.reset_index()
    filled_list.append(group)

df = pd.concat(filled_list, ignore_index=True)

In [ ]:
len(df)    

In [ ]:
df.head(20)

# Adding Technical Indicators

In [ ]:
df['daily_return'] = df.groupby('Symbol')['close'].pct_change()
df['weekly_return'] = df.groupby('Symbol')['close'].pct_change(7)
df['monthly_return'] = df.groupby('Symbol')['close'].pct_change(30)

# Volatility: Rolling std của daily_return, weekly_return, monthly_return
df['daily_volatility'] = df['daily_return'].rolling(window=5, min_periods=1).std()
df['weekly_volatility'] = df['weekly_return'].rolling(window=21, min_periods=1).std()
df['monthly_volatility'] = df['monthly_return'].rolling(window=63, min_periods=1).std()

# Liquidity: Rolling mean của Volume
df['daily_liquidity']  = df['volume'].rolling(window=5, min_periods=1).mean()
df['weekly_liquid#ity'] = df['volume'].rolling(window=21, min_periods=1).mean()
df['monthly_liquidity']= df['volume'].rolling(window=63, min_periods=1).mean()

df['high_minus_close'] = (df['high'] - df['close']) / df['open']
df['low_minus_open'] = (df['low'] - df['open']) / df['open']

df['cumulativ_return'] = df['close'] - df['close'].iloc[0] - 1

In [ ]:
def weighted_moving_average(prices, window):
    return prices.rolling(window, min_periods=1).apply(
        lambda x: np.dot(x, np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)),
        raw=True
    )
    
wma_windows = [3, 7, 14, 21, 50, 100]
for window in wma_windows:
    df[f'wma_{window}'] = weighted_moving_average(df['close'], window)

In [ ]:
df['obv'] = np.where(df['close'] > df['close'].shift(1), df['volume'], np.where(df['close'] < df['close'].shift(1), -df['volume'], 0))
df['obv'] = df['obv'].cumsum()  

In [ ]:
def RSI(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rsi = 100 * avg_gain / (avg_gain + avg_loss)
    return rsi

df['rsi_6'] = RSI(df['close'], 6)
df['rsi_12'] = RSI(df['close'], 12)
df['rsi_14'] = RSI(df['close'], 14)

In [ ]:
def stochastic_RSI_ex(rsi_series, period=14):
    min_rsi = rsi_series.rolling(window=period, min_periods=1).min()
    max_rsi = rsi_series.rolling(window=period, min_periods=1).max()
    return (rsi_series - min_rsi) / (max_rsi - min_rsi)

df['stoch_rsi_6'] = stochastic_RSI_ex(df['rsi_6'], 6)
df['stoch_rsi_12'] = stochastic_RSI_ex(df['rsi_12'], 12)
df['stoch_rsi_14'] = stochastic_RSI_ex(df['rsi_14'], 14)

In [ ]:
sma_windows = [3, 7, 14, 21, 50, 100]
for window in sma_windows:
    df[f'sma_{window}'] = df['close'].rolling(window=window, min_periods=1).mean()

In [ ]:
df['ema_6'] = df['close'].ewm(span=6, adjust=False).mean()
df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()

In [ ]:
def compute_ATR_wilder(df, period=14):
    high = df['high']
    low = df['low']
    close = df['close']
    
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/period, adjust=False).mean()
    
    return atr

df['atr_14'] = compute_ATR_wilder(df, period=14)

In [ ]:
def MFI(df, period=14):
    typical_price = (df['high'] + df['low'] + df['close'] / 3)
    mf = typical_price * df['volume']
    delta_tp = typical_price.diff()
    positive_mf = mf.where(delta_tp > 0, 0)
    negative_mf = mf.where(delta_tp < 0, 0).abs()
    pos_mf_sum = positive_mf.rolling(window=period, min_periods=period).sum()
    neg_mf_sum = negative_mf.rolling(window=period, min_periods=period).sum()
    mfi = 100 - (100 / (1 + pos_mf_sum / neg_mf_sum))
    return mfi

df['mfi_14'] = MFI(df, period=14)

In [ ]:
def ADX(df, period):
    up_move = df['high'] - df['high'].shift(1)
    down_move = df['low'].shift(1) - df['low']
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    tr = pd.concat([
        (df['high'] - df['low']),
        (df['high'] - df['close'].shift(1)).abs(),
        (df['low'] - df['close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(window=period, min_periods=period).mean()
    plus_di = 100 * (pd.Series(plus_dm).rolling(window=period, min_periods=period).sum() / atr)
    minus_di = 100 * (pd.Series(minus_dm).rolling(window=period, min_periods=period).sum() / atr)
    dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=period, min_periods=period).mean()
    return adx

df['adx_14'] = ADX(df, period=14)
df['adx_20'] = ADX(df, period=20)

In [ ]:
df['mom_1'] = df['close'] - df['close'].shift(1)
df['mom_3'] = df['close'] - df['close'].shift(3)

In [ ]:
def CCI(df, period):
    tp = (df['high'] + df['low'] + df['close']) / 3
    sma_tp = tp.rolling(window=period, min_periods=period).mean()
    md = tp.rolling(window=period, min_periods=period).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    cci = (tp - sma_tp) / (0.15 * md)
    return cci

df['cci_12'] = CCI(df, period=12)
df['cci_20'] = CCI(df, period=20)

In [ ]:
df['rocr_3'] = (df['close'] / df['close'].shift(3)) * 100
df['rocr_12'] = (df['close'] / df['close'].shift(12)) * 100

In [ ]:
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['out_macd'] = ema12 - ema26
df['out_macd_signal'] = df['out_macd'].ewm(span=9, adjust=False).mean()
df['out_macd_hist'] = df['out_macd'] - df['out_macd_signal']

In [ ]:
highest_high = df['high'].rolling(window=10, min_periods=10).max()
lowest_low = df['low'].rolling(window=10, min_periods=10).min()
df['willr'] = ((highest_high - df['close']) / (highest_high - lowest_low)) * 100

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
def TSF(series, period):
    def linreg(x):
        idx = np.arange(len(x))
        slope, intercept = np.polyfit(idx, x, 1)
        return slope * len(x) + intercept
    return series.rolling(window=period, min_periods=period).apply(linreg, raw=True)

df['tsf_10'] = TSF(df['close'], 10)
df['tsf_20'] = TSF(df['close'], 20)

In [ ]:
ema1 = df['close'].ewm(span=15, adjust=False).mean()
ema2 = ema1.ewm(span=15, adjust=False).mean()
ema3 = ema2.ewm(span=15, adjust=False).mean()
df['trix'] = (ema3 - ema3.shift(1)) / ema3.shift(1) * 100

In [ ]:
df['bbandsmiddle'] = df['close'].rolling(window=21, min_periods=21).mean()
rolling_std = df['close'].rolling(window=21, min_periods=21).std()
df['bbandsupper'] = df['bbandsmiddle'] + 2 * rolling_std
df['bbandslower'] = df['bbandsmiddle'] - 2 * rolling_std

In [ ]:
# Fill missing values with forward and backward fill
df = df.ffill().bfill()

In [ ]:
df.columns

In [ ]:
len(df.columns)

In [ ]:
len(df)

In [ ]:
symbol = 'FPT'

In [ ]:
df.to_csv(f'new/processed_{symbol}.csv', index=False)

# The End